# Live streaming, the live UI, and sub-agents — end to end

A real agent, a real model (`bedrock-mantle/google.gemma-4-26b-a4b`), real
tools against real files on disk. Nothing here is mocked, and nothing is a
picture of what the output would look like — every panel below is drawn from
events the runtime actually emitted.

| § | What you'll see |
|---|---|
| 1 | Setup — the provider, this repo, and a real workspace on disk |
| 2 | **Token by token** — the raw `text_delta` feed, printing as it arrives |
| 3 | Every call, its real arguments, its real output — *interleaved with the tokens* |
| 4 | **The live UI panel** — the chat card, redrawing in the cell as it works |
| 5 | The tree — live, alternating, and deep. Plus `run_code` writing and running code |
| 6 | **The timeline** — the JSON your frontend draws, and a Markdown report |
| 7 | **Sub-agents in depth** — a real delegated loop, its work streaming up |
| 8 | **Automatic delegation** — the agent reaches for sub-agents unprompted |
| 9 | Approvals — a write that blocks, a send that defers, then auto-approval |
| 10 | SSE frames — the same run, framed for a browser |
| 11 | A shareable HTML transcript |

Each section is independent after §1. They make real Bedrock calls, so they
cost money and take a few seconds each.

## 1 · Setup

Three things, in order: put **this repo** on the path (a notebook's cwd is
`notebooks/`, so an older `shipit_agent` installed elsewhere would win),
register the `bedrock-mantle` provider, and build a workspace with real
files in it.

In [1]:
import importlib
import sys
from pathlib import Path


def repo_root(start: Path) -> Path:
    """Walk up to the repo root.

    A notebook's cwd is `notebooks/`, which matters twice: file tools resolve
    one directory too deep, and `import shipit_agent` can resolve to a
    different (older) copy than the one you are editing.
    """
    for candidate in [start, *start.parents]:
        if (candidate / "shipit_agent" / "__init__.py").exists():
            return candidate
    raise RuntimeError(f"Could not locate the shipit_agent repo from {start}")


REPO = repo_root(Path.cwd())
sys.path.insert(0, str(REPO))
for name in [n for n in sys.modules if n.startswith("shipit_agent")]:
    del sys.modules[name]  # so a stale copy cannot survive a re-run

import shipit_agent

# If this assertion fires, the kernel is running some other shipit_agent.
loaded = Path(shipit_agent.__file__).resolve().parent.parent
assert loaded == REPO.resolve(), f"wrong shipit_agent loaded: {loaded}"

print("repo    :", REPO)
print("package :", shipit_agent.__file__)
print("version :", shipit_agent.__version__)

repo    : /Users/rahulraj/Documents/MYWORK/ai_developer/others/shipit_agent
package : /Users/rahulraj/Documents/MYWORK/ai_developer/others/shipit_agent/shipit_agent/__init__.py
version : 1.1.0


In [2]:
import importlib.util

PROVIDER = Path(
    "/Users/rahulraj/Documents/MYWORK/AFTDRK/CACHE/DRK_CACHE_BACK"
    "/drk_cache/llm/bedrock_mantle_provider.py"
)
MODEL = "bedrock-mantle/google.gemma-4-26b-a4b"

spec = importlib.util.spec_from_file_location("bedrock_mantle_provider", PROVIDER)
module = importlib.util.module_from_spec(spec)
sys.modules["bedrock_mantle_provider"] = module
spec.loader.exec_module(module)
module.ensure_registered()

from shipit_agent.llms import LiteLLMChatLLM


def llm():
    """A fresh adapter per agent — keeps each section independent."""
    return LiteLLMChatLLM(model=MODEL)


import boto3
print("identity:", boto3.client("sts").get_caller_identity()["Arn"])
print("model   :", MODEL)

identity: arn:aws:iam::275210565507:user/0x99-bedrock
model   : bedrock-mantle/google.gemma-4-26b-a4b


### A real workspace

The scenario is the one from the reference UI: **RSVP intake**. Emails land
in an inbox, a guest list needs updating. These are real files in a real
temp directory — the tools below read and write them for real.

In [3]:
import tempfile
import textwrap

WORKSPACE = Path(tempfile.mkdtemp(prefix="rsvp_"))
(WORKSPACE / "inbox").mkdir()

EMAILS = {
    "msg-1.eml": """
        From: jordan@acme.com
        Subject: Re: Spring Summit — RSVP

        Jordan Lee will attend, bringing one guest. Vegetarian, please.
    """,
    "msg-2.eml": """
        From: sam@globex.io
        Subject: Re: Spring Summit — RSVP

        Sam Osei here. Still trying to move a conflict — mark me as maybe.
    """,
    "msg-3.eml": """
        From: priya@initech.dev
        Subject: Re: Spring Summit — RSVP

        Priya Raman: sadly I can't make it this year. Next time!
    """,
}
for name, body in EMAILS.items():
    (WORKSPACE / "inbox" / name).write_text(textwrap.dedent(body).strip() + "\n")

(WORKSPACE / "guests.csv").write_text(
    "name,email,status,plus_ones\n"
    "Dana Kim,dana@northwind.co,confirmed,0\n"
    "Luis Marin,luis@umbrella.co,confirmed,1\n"
)

print("workspace:", WORKSPACE)
for path in sorted(WORKSPACE.rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(WORKSPACE))

workspace: /var/folders/bd/pq4lv0q52pv59m8pthkn60sc0000gn/T/rsvp_zs6bux4i
   guests.csv
   inbox/msg-1.eml
   inbox/msg-2.eml
   inbox/msg-3.eml


In [4]:
from shipit_agent import Agent
from shipit_agent.builtins import get_builtin_tool_map
from shipit_agent.permissions import PermissionDecision, PermissionEngine

TOOLS = get_builtin_tool_map(llm=None, project_root=str(WORKSPACE))

READERS = [TOOLS[n] for n in ("read_file", "glob_files", "grep_files")]
WRITERS = [*READERS, TOOLS["write_file"]]

# `deny` outranks `allow`, so `deny=["*"]` would deny the readers too. To mean
# "these and nothing else", allow-list them and flip the default.
READ_ONLY = PermissionEngine(
    allow=["read_file", "glob_files", "grep_files"],
    default_decision=PermissionDecision.DENY,
)
READ_WRITE = PermissionEngine(
    allow=["read_file", "glob_files", "grep_files", "write_file", "sub_agent"],
    default_decision=PermissionDecision.DENY,
)

from shipit_agent.prompts.default_agent_prompt import DEFAULT_AGENT_PROMPT

# Appended to the default prompt rather than replacing it — the default is
# what tells the model how to use tools at all.
TRIAGE = DEFAULT_AGENT_PROMPT + (
    "\n\nYou process event RSVPs. The inbox is at inbox/*.eml and the guest "
    "list is guests.csv. Paths are relative to the workspace root. Be brief."
)

print("tools :", sorted(TOOLS)[:8], "…")
print("policy: read anything in the workspace; write only where allowed")

tools : ['ask_user', 'ask_user_async', 'bash', 'build_artifact', 'build_document', 'build_prompt', 'confluence', 'connections'] …
policy: read anything in the workspace; write only where allowed


## 2 · Token by token

The lowest level there is: `agent.stream()` yields a `text_delta` for every
chunk the provider sends. This is the loop to copy if you are wiring your own
UI — the tokens arrive here, not anywhere else.

If your streaming loop looks like it isn't streaming, check that it has a
`text_delta` branch. A loop that only handles `tool_called` / `tool_completed`
will sit silent through the entire answer and then print nothing.

In [5]:
agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=6)

deltas = 0
for event in agent.stream(
    "Read inbox/msg-1.eml and say in two sentences who replied and what they said."
):
    if event.type == "text_delta":
        print(event.payload["chunk"], end="", flush=True)   # ← the tokens
        deltas += 1
    elif event.type == "tool_called":
        print(f"\n[{event.payload['tool']}] ", end="", flush=True)

print(f"\n\n{deltas} text deltas — the answer arrived in {deltas} pieces.")


[read_file] 

Jordan

 Lee

 replied

 to

...

 a

 vegetarian

 meal

.



22 text deltas — the answer arrived in 22 pieces.


## 3 · Every call, its arguments, its output — with the tokens in place

The same feed, printed in full: the exact arguments the model chose, the
exact bytes that came back, and the prose streaming between them. This is the
whole run with nothing hidden.

In [6]:
agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=8)

for event in agent.stream(
    "Use glob_files with pattern 'inbox/*.eml' to list the inbox, then "
    "read_file each message, then tell me who is attending."
):
    kind = event.type
    payload = event.payload

    if kind == "text_delta":
        print(payload["chunk"], end="", flush=True)

    elif kind == "tool_called":
        args = ", ".join(f"{k}={v!r}" for k, v in payload["arguments"].items())
        print(f"\n\n→ {payload['tool']}({args[:120]})")

    elif kind == "tool_completed":
        body = str(payload.get("output", ""))
        lines = body.splitlines()
        print(f"← {payload['duration_ms']:.0f}ms · {len(body)} bytes")
        for line in lines[:6]:
            print(f"    {line[:100]}")
        if len(lines) > 6:
            print(f"    … {len(lines) - 6} more lines")
        print()

    elif kind == "tool_failed":
        print(f"\n✗ {payload.get('error', '')[:160]}")

    elif kind == "tool_denied":
        print(f"\n⊘ blocked: {payload.get('reason', '')[:140]}")

    elif kind == "usage_tick":
        pass  # tokens so far — §4 puts these in the footer



→ glob_files(pattern='inbox/*.eml')
← 2ms · 47 bytes
    inbox/msg-1.eml
    inbox/msg-2.eml
    inbox/msg-3.eml





→ read_file(path='inbox/msg-1.eml')
← 3ms · 148 bytes
        1: From: jordan@acme.com
        2: Subject: Re: Spring Summit — RSVP
        3: 
        4: Jordan Lee will attend, bringing one guest. Vegetarian, please.





→ read_file(path='inbox/msg-2.eml')
← 2ms · 149 bytes
        1: From: sam@globex.io
        2: Subject: Re: Spring Summit — RSVP
        3: 
        4: Sam Osei here. Still trying to move a conflict — mark me as maybe.





→ read_file(path='inbox/msg-3.eml')
← 2ms · 143 bytes
        1: From: priya@initech.dev
        2: Subject: Re: Spring Summit — RSVP
        3: 
        4: Priya Raman: sadly I can't make it this year. Next time!



The

...

marked

 as

 "

maybe

")

## 4 · The live UI panel

The same events, drawn as the chat card from the reference UI — and it
**redraws in place** while the run is happening. Tokens land with a caret,
tool rows appear in-flight and settle, every call folds away with its real
output behind it, and the footer counts tokens.

Click any `✓ read_file · 4ms — 12 lines` line to open the real output.

In [7]:
from shipit_agent.narrate import watch

agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=8)

answer = watch(
    agent,
    "Read every file in inbox/ and guests.csv, then tell me which of the "
    "three people who replied are NOT yet on the guest list.",
    title="RSVP intake",
    output_limit=None,      # every byte a tool returned, in full
)

The panel is a renderer, not a magic trick — `render_chat_html(events)` gives
you the same card for a run that already finished, as an HTML fragment you
can put in a page.

In [8]:
from IPython.display import HTML

from shipit_agent.narrate import render_chat_html

agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=5)
events = list(agent.stream("How many people are on guests.csv right now?"))

HTML(render_chat_html(events, model=MODEL, title="Guest count", output_limit=None))

## 5 · The tree — the shape of the run

Prose is what the agent *said*; the tree is what it *did*. Every call named,
its status beside it, and the decisions marked where they happened.

In [9]:
agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=8)

agent.run_live(
    "Use grep_files with pattern 'attend' and path 'inbox' to find the "
    "positive replies, then read the file it matched. One sentence.",
    style="tree",
)

Agent started
│
├─ Tool group: Searched for attend, read inbox/msg-1.eml
│  ├─ grep_files                                    completed  8ms
│  └─ read_file                                     completed  1ms
│
└─ Final answer
   The positive reply is found in `inbox/msg-1.eml`, which states: "Jordan Lee will attend,
   bringing one guest. Vegetarian, please."

7,062 tokens · bedrock-mantle/google.gemma-4-26b-a4b


'The positive reply is found in `inbox/msg-1.eml`, which states: "Jordan Lee will attend, bringing one guest. Vegetarian, please."'

### The tree, live

`watch(..., shape="tree")` draws the same tree **in the cell, as it happens**:
groups appear as calls start, statuses flip from `running` to `completed`
where they stand, and the trunk stays open — `├─ working…` — until the run
ends. Drawing the final corner before the agent has finished would be a
claim, not a picture.

On a terminal, `agent.run_live(style="tree")` now does the same thing: it
redraws in place while the run proceeds, then erases the draft and writes one
clean tree into your scrollback.

In [10]:
from shipit_agent.narrate import watch

agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=8)

watch(
    agent,
    "Read guests.csv, then read every file in inbox/, then say in one line "
    "who still needs adding.",
    title="RSVP intake",
    shape="tree",
)

'Jordan Lee and Sam Osei still need adding.'

### The alternating shape — group, decision, group, answer

The shape from the mock-up — *tool group → decision → tool group → decision →
answer* — is what the tree draws **when the model narrates between batches of
work**. Prose is what breaks a work run; that rule is the whole grouping
model.

Gemma, inside one turn, usually does not narrate: it fires every call it has
planned and then answers, which renders as one group and one answer. Across a
**conversation** it narrates constantly, because each turn ends with a
sentence. So this is where the alternation shows up honestly — and a
conversation is also what a UI actually displays.

In [11]:
from shipit_agent.narrate import render_tree
from shipit_agent.tools.code_execution import CodeExecutionTool

(WORKSPACE / "venue.txt").write_text("Capacity: 3 seats.\n")

# run_code executes real code in a real subprocess. Point its workspace at
# our files, or the script it writes will not be able to see them.
runner = CodeExecutionTool(workspace_root=WORKSPACE, timeout_seconds=20)

stepwise = Agent(
    llm=llm(), tools=[*READERS, runner],
    permissions=PermissionEngine(
        allow=["read_file", "glob_files", "grep_files", "run_code"],
        default_decision=PermissionDecision.DENY),
    prompt=TRIAGE, auto_use_skills=False, max_iterations=10,
)

chat = stepwise.chat_session(session_id="venue")
conversation = []
for turn in [
    "Read guests.csv and tell me who is confirmed.",
    "Now read venue.txt — what is the capacity?",
    "Write a Python script with run_code that prints whether the confirmed "
    "guests plus their plus-ones fit in the venue. Give me the verdict.",
]:
    conversation.extend(chat.send(turn).events)

print(render_tree(conversation, model="gemma-4-26b"))

Agent started
│
├─ Tool group: Read guests.csv
│  └─ read_file                                     completed  2ms
│
├─ Decision
│  The confirmed guests are:
│  * Dana Kim
│  * Luis Marin
│
├─ Tool group: Read venue.txt
│  └─ read_file                                     completed  3ms
│
├─ Decision
│  The capacity is 3 seats.
│
├─ Tool group: Ran code import csv
│  └─ run_code                                      completed  41ms
│
└─ Final answer
   The confirmed guests and their plus-ones (3 total) fit exactly within the venue capacity (3
   seats).
   
   **Verdict: Yes, they fit.**

6,985 tokens · gemma-4-26b



### The agent writes code, and runs it

`run_code` is not a sandbox toy: it writes the model's code to a file and
executes it in a real subprocess (Docker too, with `sandbox=True`). Below,
with `detail=True`, you can read the exact program it wrote and the exact
stdout it got — including any script that failed and what it did next.

In [12]:
coder = Agent(
    llm=llm(), tools=[*READERS, runner],
    permissions=PermissionEngine(
        allow=["read_file", "glob_files", "grep_files", "run_code"],
        default_decision=PermissionDecision.DENY),
    prompt=TRIAGE, auto_use_skills=False, max_iterations=8,
)

events = list(coder.stream(
    "Read guests.csv, then write a short Python script and run it with "
    "run_code that prints how many guests are confirmed and how many "
    "plus-ones they bring in total. Say what the script printed."
))

print(render_tree(events, model="gemma-4-26b", detail=True, output_lines=8))

Agent started
│
├─ Tool group: Read guests.csv, ran code import csv
│  ├─ read_file                                     completed  2ms
│  │  ↳ path='guests.csv'
│  │    1: name,email,status,plus_ones
│  │    2: Dana Kim,dana@northwind.co,confirmed,0
│  │    3: Luis Marin,luis@umbrella.co,confirmed,1
│  └─ run_code                                      completed  59ms
│     ↳ language='python', code="import csv\n\nconfirmed_count = 0\ntotal_plus_ones = 0\n\nwith open('guests.csv', mode='r') as f:\n    reader = csv.DictReader(f)\n    
│       exit_code: 0
│       stdout:
│       Confirmed guests: 2
│       Total plus-ones: 1
│       stderr:
│
└─ Final answer
   The script printed:
   ```
   Confirmed guests: 2
   Total plus-ones: 1
   ```

9,144 tokens · gemma-4-26b



Two things that run is honest about, and both are worth seeing:

- **`run_code` runs in its own workspace.** A script that opens `guests.csv`
  finds nothing unless that workspace is where the file lives — which is why
  `workspace_root=WORKSPACE` is passed above. Left at its default, the first
  script fails with `FileNotFoundError`, and you can watch the agent recover.
- **Failures are shown, not smoothed over.** `exit_code: 1` and the traceback
  are in the tree exactly as the tool returned them.

### The deep tree

`detail=True` opens every call: what it was called with, and the first lines
of what came back. This is the view for "why did it do *that*".

In [13]:
from shipit_agent.narrate import render_tree

agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=8)

events = list(agent.stream(
    "Read guests.csv and inbox/msg-2.eml. Is Sam already on the list?"
))

print(render_tree(events, model="gemma-4-26b", detail=True, output_lines=4))

Agent started
│
├─ Tool group: Read 2 files
│  ├─ read_file                                     completed  1ms
│  │  ↳ path='guests.csv'
│  │    1: name,email,status,plus_ones
│  │    2: Dana Kim,dana@northwind.co,confirmed,0
│  │    3: Luis Marin,luis@umbrella.co,confirmed,1
│  └─ read_file                                     completed  2ms
│     ↳ path='inbox/msg-2.eml'
│       1: From: sam@globex.io
│       2: Subject: Re: Spring Summit — RSVP
│       3: 
│       4: Sam Osei here. Still trying to move a conflict — mark me as maybe.
│
└─ Final answer
   No, Sam is not on the guest list. The guest list contains Dana Kim and Luis Marin, while
   `inbox/msg-2.eml` is an email from Sam Osei (`sam@globex.io`).

7,077 tokens · gemma-4-26b



## 6 · The timeline — what your frontend draws

The runtime's own events are the wrong shape for a UI: too many, too fine.
`timeline()` translates them into the four things a UI actually draws — a
summary, groups of tool calls, the decisions between them, and the answer.

Every step is a plain JSON dict, so it goes onto a socket unchanged:

```python
for step in stream_timeline(agent, prompt):
    await websocket.send_json(step)
```

In [14]:
import json

from shipit_agent.narrate import stream_timeline

agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=8)

steps = []
for step in stream_timeline(
    agent,
    "Read inbox/msg-3.eml, then check guests.csv. Should Priya be added?",
):
    steps.append(step)
    shown = {k: v for k, v in step.items() if k != "type"}
    print(f"{step['type']:<22} {json.dumps(shown, default=str)[:110]}")

run_started            {"goal": "Read inbox/msg-3.eml, then check guests.csv. Should Priya be added?"}


tool_group_started     {"group_id": "g1", "title": "Reading inbox/msg-3.eml"}
tool_call_started      {"tool_call_id": "call_1_1", "group_id": "g1", "tool_name": "read_file", "input": {"path": "inbox/msg-3.eml"}}
tool_call_completed    {"tool_call_id": "call_1_1", "group_id": "g1", "tool_name": "read_file", "status": "completed", "duration_ms":


tool_call_started      {"tool_call_id": "call_2_1", "group_id": "g1", "tool_name": "read_file", "input": {"path": "guests.csv"}}
tool_call_completed    {"tool_call_id": "call_2_1", "group_id": "g1", "tool_name": "read_file", "status": "completed", "duration_ms":


tool_group_completed   {"group_id": "g1", "tool_calls": 2}


final_response         {"status": "completed", "content": "No, Priya should not be added. The email from `priya@initech.dev` states t


### The same run as a report

`render_markdown()` prints the timeline as a document — a PR comment, a run
log, an audit trail. Same data, no UI required.

In [15]:
from IPython.display import Markdown

from shipit_agent.narrate import render_markdown

agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=6)
report_events = list(agent.stream(
    "Read guests.csv, then read inbox/msg-1.eml. Is Jordan already listed?"
))

Markdown(render_markdown(report_events))

## Agent Run

**Goal:** Read guests.csv, then read inbox/msg-1.eml. Is Jordan already listed?

---

### 1. Tool calls

#### Reading guests.csv

##### `read_file`

**Input**

```json
{
  "path": "guests.csv"
}
```

**Status:** Completed  
**Duration:** 2 ms

**Result**

```json
"    1: name,email,status,plus_ones\n    2: Dana Kim,dana@northwind.co,confirmed,0\n    3: Luis Marin,luis@umbrella.co,confirmed,1"
```

##### `read_file`

**Input**

```json
{
  "path": "inbox/msg-1.eml"
}
```

**Status:** Completed  
**Duration:** 2 ms

**Result**

```json
"    1: From: jordan@acme.com\n    2: Subject: Re: Spring Summit — RSVP\n    3: \n    4: Jordan Lee will attend, bringing one guest. Vegetarian, please."
```

---

### 2. Final response

No, Jordan is not listed in `guests.csv`.

**Run status:** Successful  
**Tool calls:** 2  
**Total tool duration:** 0.00 seconds

**Tokens:** 7,038


## 7 · Sub-agents, in depth

A sub-agent is not a prompt trick — it is a **real agent**: its own context,
its own tools, its own loop, running to its own conclusion and reporting back
a single answer. The parent's context never sees the three emails; it sees
one summary per delegation.

The real-world shape: the lead agent triages, and delegates the reading of
each message to a fresh worker. Three inboxes, three workers, one context.

Every tool call a child makes is streamed up to the parent as a
`sub_agent_event`, so nothing it does is invisible.

In [16]:
from shipit_agent.tools.sub_agent import SubAgentTool

sub_agent = SubAgentTool(
    llm=llm(),
    tools=READERS,          # the children get the readers, and nothing else
    max_iterations=6,
    max_workers=3,
)

lead = Agent(
    llm=llm(),
    tools=[*READERS, sub_agent],
    permissions=READ_WRITE,
    prompt=TRIAGE + (
        " Delegate the reading of each individual email to a sub-agent, "
        "then combine what they report."
    ),
    auto_use_skills=False,
    max_iterations=8,
)

PROMPT = (
    "For each of inbox/msg-1.eml, inbox/msg-2.eml and inbox/msg-3.eml, "
    "delegate to a sub_agent with task='Read <path> and report the sender "
    "name, email and RSVP status in one line'. Then list all three results."
)

### Watch the delegation live

`sub_agent_event` carries the child's own events. Below, the parent's work is
flush left and each child's is indented under the task it was given — so a
nested `read_file` can never be mistaken for the parent reading a file.

In [17]:
depth = {}

for event in lead.stream(PROMPT):
    payload = event.payload

    if event.type == "tool_called" and payload["tool"] == "sub_agent":
        task = str(payload["arguments"].get("task", ""))[:70]
        print(f"\n▸ delegating: {task}")

    elif event.type == "sub_agent_event":
        inner_kind = payload.get("inner_type")
        inner = payload.get("inner") or {}
        agent_name = payload.get("agent", "sub-agent")
        if inner_kind == "tool_called":
            args = ", ".join(f"{k}={v!r}" for k, v in
                             (inner.get("arguments") or {}).items())
            print(f"     └ [{agent_name}] {inner.get('tool')}({args[:70]})")
        elif inner_kind == "tool_completed":
            body = str(inner.get("output", ""))
            print(f"       ← {len(body)} bytes")

    elif event.type == "tool_completed" and payload.get("tool") == "sub_agent":
        print(f"  ✓ child finished in {payload['duration_ms']:.0f}ms")
        print(f"    {str(payload.get('output', ''))[:200]}")

    elif event.type == "text_delta":
        print(event.payload["chunk"], end="", flush=True)


▸ delegating: Read inbox/msg-1.eml and report the sender name, email and RSVP status


     └ [sub-agent] read_file(path='inbox/msg-1.eml')
       ← 148 bytes


  ✓ child finished in 1754ms
    Jordan Lee (jordan@acme.com) - Attending



▸ delegating: Read inbox/msg-2.eml and report the sender name, email and RSVP status


     └ [sub-agent] read_file(path='inbox/msg-2.eml')
       ← 149 bytes


...

.

dev

):

 Not

 attending

### The same delegation, in the live panel

Delegated work gets its own attributed row — the child's name, the task, and
its calls — rather than being folded into the parent's.

In [18]:
lead2 = Agent(
    llm=llm(),
    tools=[*READERS, SubAgentTool(llm=llm(), tools=READERS, max_iterations=5)],
    permissions=READ_WRITE,
    prompt=TRIAGE,
    auto_use_skills=False,
    max_iterations=6,
)

watch(
    lead2,
    "Delegate to a sub_agent with task='Read inbox/msg-1.eml and report the "
    "sender and RSVP status', then tell me what it found.",
    title="Delegated RSVP read",
    output_limit=None,
)

'The sub-agent found the following:\n\n*   **Sender:** jordan@acme.com\n*   **RSVP status:** Will attend (with one guest)'

### What a sub-agent actually costs

The point of delegating is context: the parent pays for one summary, not for
everything the child read.

In [19]:
result = lead2.run(
    "Delegate to a sub_agent with task='Read inbox/msg-2.eml and report the "
    "sender and RSVP status'. Report what came back."
)

child_calls = [e for e in result.events if e.type == "sub_agent_event"
               and e.payload.get("inner_type") == "tool_called"]
parent_calls = [e for e in result.events if e.type == "tool_called"]

print("parent tool calls :", [e.payload["tool"] for e in parent_calls])
print("child tool calls  :", [(e.payload.get("agent"),
                               (e.payload.get("inner") or {}).get("tool"))
                              for e in child_calls])
print("parent messages   :", len(result.messages))
print("answer            :", result.output[:220])

parent tool calls : ['sub_agent']
child tool calls  : [('sub-agent', 'read_file')]
parent messages   : 5
answer            : The sub-agent reported the following from `inbox/msg-2.eml`:

*   **Sender:** sam@globex.io (Sam Osei)
*   **RSVP Status:** Maybe


## 8 · The agent delegates on its own

Everything in §7 said the word `sub_agent` in the prompt. Real work does not.
`Agent(delegation=True)` makes delegation something the agent reaches for
without being told:

- a `sub_agent` tool is **guaranteed to exist** — built from this agent's own
  LLM and its read-only tools
- the task is **sized by a model**, not a keyword list: one cheap cached
  question — does this split into independent pieces, and how many? — with a
  structural count (enumerated lists, named targets, stated quantities) as
  both the fallback and the floor
- the directive lands on the **task**, not the system prompt

That last one is not a style choice. Same three reports, same model: in the
system prompt Gemma delegated **zero** times; the same words appended to the
task, **six**. A small model reads the system prompt as background and the
task as instructions.

What it will not do is delegate behind the model's back. The runtime cannot
know which parts of a task are independent — guessing would spawn children
working on halves of one indivisible problem.

In [20]:
from shipit_agent.delegation import DelegationPolicy, StructuralAssessor

# What the policy makes of a task, before any model is asked.
for task in [
    "Read guests.csv and tell me the count.",
    "Summarize inbox/msg-1.eml, inbox/msg-2.eml and inbox/msg-3.eml.",
    "Go through:\n1. msg-1\n2. msg-2\n3. msg-3\n4. guests.csv",
]:
    advice = DelegationPolicy(assessor=StructuralAssessor()).assess(task)
    verdict = f"delegate ×{advice.items}" if advice else "do it yourself"
    print(f"{verdict:<18} {task.splitlines()[0][:56]}")
    for reason in advice.reasons:
        print(f"{'':<18} · {reason}")

do it yourself     Read guests.csv and tell me the count.
delegate ×3        Summarize inbox/msg-1.eml, inbox/msg-2.eml and inbox/msg
                   · 3 concrete targets are named
delegate ×4        Go through:
                   · the task enumerates 4 items


### Now watch it happen — the prompt never mentions sub-agents

In [21]:
smart = Agent(
    llm=llm(), tools=READERS, permissions=READ_WRITE,
    prompt=TRIAGE, auto_use_skills=False, max_iterations=8,
    delegation=True,          # ← the whole feature
)

result = smart.run(
    "Summarize inbox/msg-1.eml, inbox/msg-2.eml and inbox/msg-3.eml in one "
    "line each, then say which of them is not attending."
)

parent = [e.payload["tool"] for e in result.events if e.type == "tool_called"]
children = [
    (e.payload.get("agent"), (e.payload.get("inner") or {}).get("tool"))
    for e in result.events
    if e.type == "sub_agent_event" and e.payload.get("inner_type") == "tool_called"
]
print("parent calls :", parent)
print("delegations  :", parent.count("sub_agent"))
print("child calls  :", children)
print()
print(result.output[:400])

parent calls : ['sub_agent', 'sub_agent', 'sub_agent', 'sub_agent', 'sub_agent']
delegations  : 5
child calls  : [('sub-agent', 'read_file'), ('sub-agent', 'read_file'), ('sub-agent', 'read_file'), ('sub-agent', 'read_file')]

inbox/msg-1.eml: Jordan Lee will attend the Spring Summit with one guest and requests a vegetarian meal.
inbox/msg-2.eml: Sam Osei is marking himself as a "maybe" for the Spring Summit due to a potential scheduling conflict.
inbox/msg-3.eml: Priya Raman replied that she cannot attend the Spring Summit this year.

Priya Raman (inbox/msg-3.eml) is not attending.


The parent never opened a file. Three emails were read in three separate
contexts, and only three one-line answers came back into this one — which is
the entire economic argument for sub-agents.

In [22]:
# The same run as a tree: delegation is visible as its own kind of branch.
from shipit_agent.narrate import render_tree

print(render_tree(result.events, model="gemma-4-26b"))

Agent started
│
├─ Tool group: Delegated 2 tasks, started 2 tasks, collected task-1
│  ├─ sub_agent                                     completed  1463ms
│  ├─ sub_agent                                     completed  2ms
│  ├─ sub_agent                                     completed  2ms
│  ├─ sub_agent                                     completed  584ms
│  └─ sub_agent                                     completed  1382ms
│
├─ Delegated: Summarize inbox/msg-1.eml in exactly one line. Return only t
│  └─ read_file                                     completed
│
├─ Delegated: Summarize inbox/msg-2.eml in exactly one line. Return only t
│  └─ read_file                                     completed
│
├─ Delegated: Summarize inbox/msg-3.eml in exactly one line. Return only t
│  ├─ read_file                                     completed
│  └─ read_file                                     completed
│
└─ Final answer
   inbox/msg-1.eml: Jordan Lee will attend the Spring Summit with one guest 

## 9 · Approvals — two different holds

Two ways a side-effecting call gets held, and they behave differently. Both
are real below.

**A. The agent stops.** `write_file` declares `await_decision=True` in its
contract: the agent is going to reason over the result, and an agent that
believes it wrote a file it never wrote will re-read it, disbelieve itself,
and undo its own work. So an `ask` on a write **blocks** — nothing is queued,
and the agent is told it needs approval before it can continue.

**B. The agent keeps going.** A fire-and-forget send — post a message, file a
ticket — has nothing to reason over: "queued" is the whole result. Those are
**deferred**: the call goes into the queue, the agent is told the truth and
carries on, and you approve the batch afterwards.


**Watch what the model says versus what the next cell shows.** Gemma reports
the row as appended; the file is untouched and nothing is queued. The block
worked — the model's own summary is wrong. That gap is exactly why writes
carry `await_decision`: an agent left running against a file it never wrote
will re-read it, disbelieve itself, and start undoing its own work.

In [23]:
from shipit_agent.approvals import ApprovalQueue

queue = ApprovalQueue()

HOLD_WRITES = PermissionEngine(
    allow=["read_file", "glob_files", "grep_files"],
    ask=["write_file"],                       # ← needs a human
    default_decision=PermissionDecision.DENY,
)

writer = Agent(
    llm=llm(), tools=WRITERS, permissions=HOLD_WRITES, approvals=queue,
    prompt=TRIAGE, auto_use_skills=False, max_iterations=6,
)

watch(
    writer,
    "Append a row for Jordan Lee (jordan@acme.com, confirmed, 1 plus one) "
    "to guests.csv using write_file.",
    title="RSVP intake — write blocked",
    output_limit=None,
)

'I cannot directly execute the `write_file` tool because it requires manual approval from you. \n\nTo complete the task, please **approve** the pending `write_file` call for `guests.csv` with the content:\n`Jordan Lee,jordan@acme.com,confirmed,1`'

In [24]:
# A. blocked, not queued — and the file is untouched.
print("queued          :", [a.title for a in queue.pending()] or "nothing")
print("guests.csv still:")
print((WORKSPACE / "guests.csv").read_text())

queued          : nothing
guests.csv still:
name,email,status,plus_ones
Dana Kim,dana@northwind.co,confirmed,0
Luis Marin,luis@umbrella.co,confirmed,1



### B · A send that really gets deferred

A tool that declares itself a `comms.send` — the same contract the built-in
`slack` tool carries. `await_decision` is off, so the queue holds it and the
run continues. It is a **real** tool: approving it writes the notice to disk,
and nothing writes it before you do.

In [25]:
from shipit_agent.tools.base import ToolOutput
from shipit_agent.tools.contracts import CONTRACTS, ToolContract

SENT = WORKSPACE / "sent.log"


class NotifyGuest:
    """Send an RSVP confirmation. Real side effect: appends to sent.log."""

    name = "notify_guest"
    description = "Send a confirmation email to a guest."
    prompt_instructions = ""
    # Same action kind as the built-in `slack` tool — one "always approve
    # sending?" answer covers both.
    contract = ToolContract(
        action_kind=CONTRACTS["slack"].action_kind,
        auto_approvable=True,
        await_decision=False,      # ← "queued" is the whole result
    )

    def schema(self):
        return {
            "type": "function",
            "function": {
                "name": self.name,
                "description": self.description,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "email": {"type": "string"},
                        "message": {"type": "string"},
                    },
                    "required": ["email", "message"],
                },
            },
        }

    # Tools take the ToolContext first, then their own named arguments.
    def run(self, context, *, email: str, message: str) -> ToolOutput:
        with SENT.open("a") as handle:
            handle.write(f"{email}: {message}\n")
        return ToolOutput(text=f"sent to {email}")


notifier = ApprovalQueue()
sender = Agent(
    llm=llm(), tools=[*READERS, NotifyGuest()], approvals=notifier,
    permissions=PermissionEngine(
        allow=["read_file", "glob_files", "grep_files"],
        ask=["notify_guest"],
        default_decision=PermissionDecision.DENY,
    ),
    prompt=TRIAGE, auto_use_skills=False, max_iterations=6,
)

watch(
    sender,
    "Use notify_guest to send jordan@acme.com the message "
    "'Your RSVP is confirmed — see you at the Spring Summit.'",
    title="RSVP intake — send held",
    output_limit=None,
)

'The confirmation message has been queued for jordan@acme.com.'

In [26]:
# Held, and nothing has been sent.
for action in notifier.pending():
    print(f"#{action.id}  {action.title}")
    print(f"     kind      : {action.kind_label}  ({action.tag})")
    print(f"     revertible: {action.contract.implements_revert}")
    print(f"     arguments : {action.arguments}")
    print()
    print("     " + str(action.description).replace("\n", "\n     ")[:360])

print()
print("sent.log exists :", SENT.exists())

#1  Notified guest jordan@acme.com
     kind      : Send a message on your behalf  (comms.send)
     revertible: False
     arguments : {'email': 'jordan@acme.com', 'message': 'Your RSVP is confirmed — see you at the Spring Summit.'}

     Run `notify_guest` with:
     
     - **email**: `jordan@acme.com`
     - **message**: `Your RSVP is confirmed — see you at the Spring Summit.`

sent.log exists : False


In [27]:
# Approve it — and only now does the send happen.
for action in list(notifier.pending()):
    notifier.approve(action.id)

print("pending after approval:", notifier.pending() or "none")
print("sent.log              :", SENT.read_text() if SENT.exists() else "(nothing)")

pending after approval: none
sent.log              : jordan@acme.com: Your RSVP is confirmed — see you at the Spring Summit.



### "Always approve this kind"

Auto-approval needs **both** signals: the tool's contract must mark the kind
auto-approvable *and* you must enable the rule for that tag. Neither alone is
enough, which is why a tool author cannot quietly widen what runs unattended.

In [28]:
auto = ApprovalQueue()
auto.enable_auto(CONTRACTS["slack"].action_kind, by="you")

SENT.unlink(missing_ok=True)
sender2 = Agent(
    llm=llm(), tools=[NotifyGuest()], approvals=auto,
    permissions=PermissionEngine(ask=["notify_guest"],
                                 default_decision=PermissionDecision.DENY),
    prompt=TRIAGE, auto_use_skills=False, max_iterations=4,
)
sender2.run(
    "Use notify_guest to send sam@globex.io the message 'Noted — maybe.'"
)

print("rule enabled :", [r.label for r in auto.rules()])
print("still pending:", auto.pending() or "none")
print("sent.log     :", SENT.read_text() if SENT.exists() else "(nothing)")

rule enabled : ['Send a message on your behalf']
still pending: none
sent.log     : sam@globex.io: Noted — maybe.



## 10 · SSE — the same run, framed for a browser

`stream_sse()` frames every event for `text/event-stream`, opening with a
generation id so a reconnecting client knows whether to keep what it already
drew.

In [29]:
agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=5)

frames = []
for frame in agent.stream_sse("How many emails are in inbox/?"):
    frames.append(frame)

print(f"{len(frames)} frames\n")
for frame in frames[:6]:
    print(frame.rstrip())
    print("--")
print("…")
print(frames[-1].rstrip())

23 frames

id: 0
event: stream_hello
data: {"type": "stream_hello", "durability": "control", "generation": "79722-1786020005881", "sequence": 0, "message": "Stream opened", "payload": {"generation": "79722-1786020005881", "discard_provisional": true}, "timestamp": 1786020088.9680471}
--
id: 1
event: run_started
data: {"type": "run_started", "durability": "canonical", "generation": "79722-1786020005881", "sequence": 1, "message": "Agent run started", "payload": {"prompt": "How many emails are in inbox/?"}, "timestamp": 1786020088.968886}
--
id: 2
event: step_started
data: {"type": "step_started", "durability": "provisional", "generation": "79722-1786020005881", "sequence": 2, "message": "LLM completion started", "payload": {"tool_count": 3, "iteration": 1}, "timestamp": 1786020088.9689338}
--
id: 3
event: usage_tick
data: {"type": "usage_tick", "durability": "provisional", "generation": "79722-1786020005881", "sequence": 3, "message": "Usage updated", "payload": {"usage": {"prompt_token

## 11 · A transcript you can send someone

One self-contained HTML file: no network, no build step, no JavaScript beyond
the disclosure toggles.

In [30]:
from shipit_agent.narrate import write_transcript

agent = Agent(llm=llm(), tools=READERS, permissions=READ_ONLY,
              prompt=TRIAGE, auto_use_skills=False, max_iterations=6)
result = agent.run("Summarize the three RSVP emails in three lines.")

path = WORKSPACE / "rsvp_run.html"
write_transcript(path, result.events, model=MODEL,
                 title="RSVP intake", prompt="Summarize the three RSVP emails")
print("wrote", path, f"({path.stat().st_size:,} bytes)")
print(result.output[:300])

wrote /var/folders/bd/pq4lv0q52pv59m8pthkn60sc0000gn/T/rsvp_zs6bux4i/rsvp_run.html (4,727 bytes)
Jordan Lee is attending with one guest and requests a vegetarian meal.
Sam Osei is a "maybe" due to a potential scheduling conflict.
Priya Raman cannot attend this year.


---

### Where each surface lives

| You want | Use |
|---|---|
| Tokens as they arrive | `agent.stream()` → `text_delta` |
| A terminal transcript | `agent.run_live(style="modern")` |
| The shape of the run | `agent.run_live(style="tree")`, `render_tree(..., detail=True)` |
| A live panel in a notebook | `watch(agent, prompt)` |
| A finished panel in a page | `render_chat_html(events)` |
| Events for a frontend | `stream_timeline(agent, prompt)` |
| A report | `render_markdown(events)` |
| A browser stream | `agent.stream_sse(prompt)` |
| A file to send someone | `write_transcript(path, events)` |
| Sub-agents without asking | `Agent(delegation=True)` |